# 01 · Ring all-reduce from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/01_ring_allreduce.ipynb) &nbsp; 📖 Concept deck: [Data parallelism](https://nilesh-salpe.github.io/visual-ai/distributed/02-data-parallelism/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** the ring all-reduce used for data-parallel gradients: a reduce-scatter phase and an all-gather phase on N simulated GPUs, with the bytes each GPU sends.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def make_grads(N, L, seed=0):
    """N ranks, each with its own gradient vector of length L (L divisible by N)."""
    r = np.random.default_rng(seed)
    return [r.normal(size=L) for _ in range(N)]

## Task 1: Reduce-scatter around a ring

Split every rank's vector into N chunks. In step s = 0 … N−2, rank r sends chunk (r − s) mod N to rank (r + 1) mod N, which **adds** it to its own copy of that chunk. All sends in a step happen at the same time, so use the values from before the step.
After N − 1 steps, rank r holds the **fully summed** chunk (r + 1) mod N.

Return `(chunks, sent)`: `chunks[r]` is rank r's list of N chunks after the phase, and `sent[r]` is the number of elements rank r sent.

In [ ]:
def reduce_scatter_ring(vectors):
    N = len(vectors)
    chunks = [list(np.split(v.astype(float).copy(), N)) for v in vectors]
    sent = [0] * N
    # TODO: N-1 steps; in each, every rank r sends chunk (r - s) % N to rank (r + 1) % N, which adds it
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_reduce_scatter_ring(f):
    for N, L in [(2, 6), (4, 12), (5, 20)]:
        g = make_grads(N, L, N); total = np.split(sum(g), N)
        chunks, sent = f(g)
        for r in range(N):
            k = (r + 1) % N
            assert np.allclose(chunks[r][k], total[k]), f"N={N}: rank {r} should hold the full sum of chunk {k}"
        assert all(s == (N - 1) * L // N for s in sent), f"N={N}: each rank sends (N-1)·L/N elements, got {sent}"

check("reduce_scatter_ring", _test_reduce_scatter_ring, reduce_scatter_ring)

## Task 2: All-gather around the ring

Now spread the summed chunks. Rank r starts owning chunk (r + 1) mod N. In step s = 0 … N−2, rank r sends chunk (r + 1 − s) mod N to rank (r + 1) mod N, which **overwrites** its copy.
After N − 1 steps every rank holds every summed chunk. Return `(chunks, sent)` as before.

In [ ]:
def all_gather_ring(chunks):
    N = len(chunks)
    chunks = [[c.copy() for c in row] for row in chunks]
    sent = [0] * N
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_all_gather_ring(f):
    N, L = 4, 12
    g = make_grads(N, L, 7); total = np.split(sum(g), N)
    rs, _ = reduce_scatter_ring_solution(g) if 'reduce_scatter_ring_solution' in globals() else _rs_ref(g)
    chunks, sent = f(rs)
    for r in range(N):
        assert np.allclose(np.concatenate(chunks[r]), sum(g)), f"rank {r} must end with the full summed vector"
    assert all(s == (N - 1) * L // N for s in sent)

def _rs_ref(vectors):
    N = len(vectors); ch = [list(np.split(v.astype(float).copy(), N)) for v in vectors]
    for s in range(N - 1):
        msgs = [((r + 1) % N, (r - s) % N, ch[r][(r - s) % N].copy()) for r in range(N)]
        for dst, k, d in msgs: ch[dst][k] = ch[dst][k] + d
    return ch, None

check("all_gather_ring", _test_all_gather_ring, all_gather_ring)

## Task 3: All-reduce = reduce-scatter + all-gather

Compose the two phases. Return `(results, sent)`: `results[r]` is the full summed vector on rank r, and `sent[r]` the total elements rank r sent.
Each rank sends $2\frac{N-1}{N}L$ elements, which is **almost independent of N**. That is why ring all-reduce scales to many GPUs in bandwidth terms.

In [ ]:
def ring_allreduce(vectors):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_ring_allreduce(f):
    for N in [2, 3, 8]:
        g = make_grads(N, 8 * N, N + 1)
        res, sent = f(g)
        assert all(np.allclose(x, sum(g)) for x in res), f"N={N}: every rank needs the full sum"
        assert all(s == 2 * (N - 1) * 8 for s in sent), f"N={N}: expected 2(N-1)/N·L = {2 * (N - 1) * 8}, got {sent}"

check("ring_allreduce", _test_ring_allreduce, ring_allreduce)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Reduce-scatter around a ring
def reduce_scatter_ring_solution(vectors):
    N = len(vectors)
    chunks = [list(np.split(v.astype(float).copy(), N)) for v in vectors]
    sent = [0] * N
    for s in range(N - 1):
        msgs = [((r + 1) % N, (r - s) % N, chunks[r][(r - s) % N].copy()) for r in range(N)]
        for r, (dst, k, data) in enumerate(msgs):
            chunks[dst][k] = chunks[dst][k] + data
            sent[r] += data.size
    return chunks, sent

check("reduce_scatter_ring (reference)", _test_reduce_scatter_ring, reduce_scatter_ring_solution)

In [ ]:
# @title Solution for Task 2: All-gather around the ring
def all_gather_ring_solution(chunks):
    N = len(chunks)
    chunks = [[c.copy() for c in row] for row in chunks]
    sent = [0] * N
    for s in range(N - 1):
        msgs = [((r + 1) % N, (r + 1 - s) % N, chunks[r][(r + 1 - s) % N].copy()) for r in range(N)]
        for r, (dst, k, data) in enumerate(msgs):
            chunks[dst][k] = data
            sent[r] += data.size
    return chunks, sent

check("all_gather_ring (reference)", _test_all_gather_ring, all_gather_ring_solution)

In [ ]:
# @title Solution for Task 3: All-reduce = reduce-scatter + all-gather
def ring_allreduce_solution(vectors):
    ch, s1 = reduce_scatter_ring_solution(vectors)
    ch, s2 = all_gather_ring_solution(ch)
    return [np.concatenate(c) for c in ch], [a + b for a, b in zip(s1, s2)]

check("ring_allreduce (reference)", _test_ring_allreduce, ring_allreduce_solution)

---
## Experiments
**Ring vs "send everything to rank 0".** With a simple cost model, time = number of sequential steps × α (latency per message) + bytes on the busiest link ÷ bandwidth. Ring: 2(N−1) steps, 2(N−1)/N·S bytes per rank. Naive gather-and-broadcast through rank 0: rank 0 receives (N−1)·S and sends (N−1)·S. Bandwidth wins for ring; latency grows with N, which is where tree algorithms help.

In [ ]:
S, alpha, bw = 2e9, 10e-6, 50e9          # 1B BF16 gradients = 2 GB; 10 µs per message; 50 GB/s per link
for N in [8, 64, 512, 4096]:
    ring = 2 * (N - 1) * alpha + 2 * (N - 1) / N * S / bw
    naive = 2 * (N - 1) * S / bw
    print(f"N={N:5d}: ring {ring * 1e3:8.1f} ms (latency part {2 * (N - 1) * alpha * 1e3:6.1f} ms)   naive via rank 0 {naive:8.1f} s")

---
## Takeaways
* Ring all-reduce = reduce-scatter (sum, each rank ends with one chunk) + all-gather (share the chunks).
* Each GPU sends ≈ 2·S regardless of N, so the bandwidth cost of data parallelism stays flat as you add GPUs.
* The latency term grows with N (2(N−1) sequential steps), so at large scale libraries switch to trees or hierarchical schemes; and frameworks overlap all-reduce with the backward pass in buckets (PyTorch DDP).